In [ ]:
# SETUPENV
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentContentFormat
from config import config
from pathlib import Path

### Check Search Index

In [ ]:
sc = SearchClient(config.ai_search.endpoint, config.ai_search.index_name, AzureKeyCredential(config.ai_search.api_key))
print("doc count:", sc.get_document_count())

### Test hybrid query see output

In [ ]:
results = sc.search(search_text="transmittal form", top=5)  # keyword baseline
for r in results:
    print("!!!!! START")
    print(r["title"], r["page"], r["filepath"], r["@search.score"], r["content"])

### Delete entire search index

In [ ]:
sic = SearchIndexClient(config.ai_search.endpoint, AzureKeyCredential(config.ai_search.api_key))

sic.delete_index(config.ai_search.index_name)
print(f"Index '{config.ai_search.index_name}' deleted")

### Document Intelligence Testing

In [ ]:
dic = DocumentIntelligenceClient(config.doc_intelligence.endpoint, AzureKeyCredential(config.doc_intelligence.api_key))
file_path = Path("../data/00 Chatbot on PDDM Info - Query List.pdf")

with open(file_path, "rb") as f:
    poller = dic.begin_analyze_document(
        model_id="prebuilt-layout",
        body=f,
        output_content_format=DocumentContentFormat.MARKDOWN,
        content_type="application/pdf",
    )
result = poller.result()

print(result.tables)

In [ ]:
print(result)